<a href="https://colab.research.google.com/github/victorbaraunaAcad/crise-saude-ufam/blob/Leonardo-branch/Trabalho01_executavel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json, time, datetime, pathlib, unicodedata
from getpass import getpass

import requests
import pandas as pd

print(f"pandas  {pd.__version__}")
print(f"requests {requests.__version__}")

pandas  2.2.3
requests 2.32.4


In [ ]:
from getpass import getpass

token = getpass("Cole o token do GitHub: ")

%cd /content

!git clone -b Leonardo-branch https://{token}@github.com/victorbaraunaAcad/crise-saude-ufam.git
%cd crise-saude-ufam

!git config user.name "8january"
!git config user.email "leonardobrandaoamarante@gmail.com"

Cole o token do GitHub: ··········
/content
Cloning into 'crise-saude-ufam'...
remote: Enumerating objects: 420, done.
remote: Counting objects: 100% (255/255), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 420 (delta 85), reused 185 (delta 29), pack-reused 165 (from 1)
Receiving objects: 100% (420/420), 192.89 MiB | 26.35 MiB/s, done.
Resolving deltas: 100% (124/124), done.
/content/crise-saude-ufam


In [ ]:
RAIZ     = pathlib.Path.cwd()
BRUTOS   = RAIZ / "dados_brutos"
TRATADOS = RAIZ / "dados_tratados"
BRUTOS.mkdir(exist_ok=True); TRATADOS.mkdir(exist_ok=True)
print("Trabalhando em:", RAIZ)

Trabalhando em: /content/crise-saude-ufam


In [ ]:
import shutil
import subprocess
import time
from pathlib import Path

RAIZ = Path.cwd()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"

def limpar_pasta(caminho_pasta: Path, nome: str):
    """Esvazia o conteúdo de um diretório sem apagar a pasta raiz."""
    if caminho_pasta.exists():
        print(f"🧹 Limpando diretório: {nome}...")
        for item in caminho_pasta.glob("*"):
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
        print(f"✓ {nome} limpo.")
    else:
        caminho_pasta.mkdir(parents=True, exist_ok=True)

def executar_notebook(nome_notebook: str) -> bool:
    """Executa um notebook .ipynb via jupyter nbconvert em segundo plano."""
    caminho = RAIZ / nome_notebook
    if not caminho.exists():
        print(f"⚠️ Arquivo não encontrado: {nome_notebook}")
        return False

    print(f"\n🚀 Executando notebook: {nome_notebook}...")
    inicio = time.time()

    cmd = [
        "jupyter", "nbconvert",
        "--to", "notebook",
        "--execute",
        "--inplace",
        str(caminho)
    ]

    resultado = subprocess.run(cmd, capture_output=True, text=True)
    tempo = time.time() - inicio

    if resultado.returncode == 0:
        print(f"✓ {nome_notebook} executado com sucesso em {tempo:.2f}s.")
        return True
    else:
        print(f"❌ Erro ao executar {nome_notebook}:")
        print(resultado.stderr)
        return False

def testar_reprodutibilidade():
    print("=== INICIANDO TESTE END-TO-END DE REPRODUTIBILIDADE ===")

    # 1. Reseta os ambientes de entrada e saída
    limpar_pasta(DIR_BRUTOS, "dados_brutos")
    limpar_pasta(DIR_TRATADOS, "dados_tratados")

    # 2. Lista completa de notebooks na ordem exata de execução
    notebooks_pipeline = [
        # Web Scraping / Aquisições específicas
        "Trabalho1_Aquisicao_Dados_Enchentes_RS_Final_.ipynb",
        "Trabalho1_Aquisicao_Dados_Estiagem_AM_Final_.ipynb",
        "Trabalho1_Aquisicao_Dados_OndaCalor_CO_Final_.ipynb",

        # Coletas gerais e DATASUS
        "coleta.ipynb",
        "coleta_DATASUS.ipynb",

        # Consolidação e tratamento (sempre por último)
        "Tratamento_e_integracao.ipynb"
    ]

    # 3. Executa a sequência do pipeline
    for nb in notebooks_pipeline:
        sucesso = executar_notebook(nb)
        if not sucesso:
            print(f"\n❌ PIPELINE INTERROMPIDO por falha no notebook: {nb}")
            return

    # 4. Checagem final de integridade
    base_parquet = DIR_TRATADOS / "base_consolidada_municipios.parquet"
    if base_parquet.exists():
        print("\n✅ PIPELINE 100% REPRODUTÍVEL! Dados coletados e base gerada do zero com sucesso.")
    else:
        print("\n❌ FALHA: A base consolidada final não foi encontrada em dados_tratados.")

if __name__ == "__main__":
    testar_reprodutibilidade()

=== INICIANDO TESTE END-TO-END DE REPRODUTIBILIDADE ===
🧹 Limpando diretório: dados_brutos...
✓ dados_brutos limpo.
🧹 Limpando diretório: dados_tratados...
✓ dados_tratados limpo.

🚀 Executando notebook: coleta.ipynb...
✓ coleta.ipynb executado com sucesso em 28.23s.

🚀 Executando notebook: coleta_DATASUS.ipynb...
❌ Erro ao executar coleta_DATASUS.ipynb:
[NbConvertApp] Converting notebook /content/crise-saude-ufam/coleta_DATASUS.ipynb to notebook
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Se